# Exercício de Fine Tuning do BERT com PyTorch 🤖🐍🔥
Neste exercício, você irá implementar o fine tuning do BERT adicionando Task Head com finalidade de classificação. Sugestões utilizando a biblioteca PyTorch.

Conforme visto em sala, o Self-attention é um componente central dos Transformers, as redes neurais que impulsionam os modelos de linguagem modernos como o BERT. Após as camadas de atenção dos modelos, podemos adicionar uma ou mais camadas com a finalidade de executar tarefas específicas.

## Contexto
Um modelo BERT é um Transformer do tipo Encoder que processa um texto de entrada gerando representações semânticas desse texto. O Self-attention permite que o modelo determine a relação entre diferentes tokens em uma sequência.

Adicionaremos, conforme visto em sala de aula, Task Head ao modelo para fazer fine tuning de classificação.



---




### [<img src="https://colab.google/static/images/icons/colab.png" width=100> OPICIONAL ] Configuração do ambiente para melhor desempenho e instalação de dependências
💡 **Obs**: Selecione um ambiente com GPU para rodar esse notebook. No Google Colab:
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
# Caso esteja no Google Colab será necessário instalar apenas as dependências abaixo
!pip install seqeval>=1.2.2
!pip install evaluate>=0.4.0

# Importando dataset

Utilizaremos o dataset Rotten Tomatoes, que contém avaliações de filmes (https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes).

In [ ]:
from datasets import load_dataset

# Preparando dados
tomatoes = load_dataset("rotten_tomatoes")
# Separando
train_data, test_data = tomatoes["train"], tomatoes["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [ ]:
# Vejamos o que tem aqui...
print(train_data)

print(train_data[:10])

print(train_data[-10:])

Dataset({
    features: ['text', 'label'],
    num_rows: 8530
})
{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .', 'effective but too-tepid biopic', 'if you sometimes like to go to the movies to have fun , wasabi is a good place to start .', "emerges as something rare , an issue movie that's so honest and keenly observed that it doesn't feel like one .", 'the film provides some great insight into the neurotic mindset of all comics -- even those who have reached the absolute top of the game .', 'offers that rare combination of entertainment and education .', 'perhaps no picture ever made has more literally showed that 

# Carregando o modelo BERT

Utilizaremos checkpoint "bert-base-uncased".

Carregaremos o tokenizer, conforme exercícios anteriores, mas também a classe `AutoModelForSequenceClassification`.

A classe `AutoModelForSequenceClassification` da biblioteca Hugging Face Transformers é uma classe automática projetada para simplificar o carregamento de modelos de Sequence Classification (Classificação de Sequência) já criando nossa Task Head.

Vejamos a diferença entre o carregamento do modelo original e o modelo com a Task Head Classification.

In [ ]:
from transformers import AutoModel, AutoModelForSequenceClassification, AutoConfig
import torch.nn as nn

# Suprime warnings
import warnings
warnings.filterwarnings('ignore')

# O checkpoint base do BERT (sem ajuste fino para tarefa específica)
MODEL_CHECKPOINT = "bert-base-uncased"
QNT_CLASSES = 2  # Definimos 2 classes (ex: positivo, negativo)

print("--- 1. Carregando APENAS o Backbone BERT (BertModel) ---")
# AutoModel carrega o modelo base (o 'backbone' ou 'corpo' do transformer)
modelo_base = AutoModel.from_pretrained(MODEL_CHECKPOINT)

# Exibe a estrutura do modelo base
print(modelo_base)

print("\n" + "="*80 + "\n")

print(f"--- 2. Carregando BERT com a Cabeça de Classificação ({QNT_CLASSES} classes) ---")

# 2.1. Criar uma configuração para 3 classes
config = AutoConfig.from_pretrained(MODEL_CHECKPOINT, num_labels=QNT_CLASSES)

# 2.2. Carregar o modelo usando AutoModelForSequenceClassification
modelo_class = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    config=config
)

# Exibe a estrutura do modelo de Classificação
print(modelo_class)

--- 1. Carregando APENAS o Backbone BERT (BertModel) ---


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

## Carregando novamente o modelo para utilização em nossa prática

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

modelo = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## 💭 Tokenizando a entrada

In [ ]:
from transformers import DataCollatorWithPadding

# Necessário para otimizar o padding no batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Definindo função de preprocessamento dos dados
def preprocessamento(itens):
   return tokenizer(itens["text"], truncation=True)

# Tokenizando dados train e test
tokenized_train = train_data.map(preprocessamento, batched=True)
tokenized_test = test_data.map(preprocessamento, batched=True)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

## Definição de métricas

In [ ]:
import numpy as np
import evaluate


def compute_metrics(eval_pred):
    """Calculate F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    load_f1 = evaluate.load("f1")
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}

# Treinamento

In [ ]:
from transformers import TrainingArguments, Trainer

# Argumentos do treinamento
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

# Instanciando o objeto "treinador"
treinador = Trainer(
   model=modelo,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)


In [ ]:
treinador.train()

Step,Training Loss
500,0.385300


TrainOutput(global_step=534, training_loss=0.38143503889162444, metrics={'train_runtime': 100.2638, 'train_samples_per_second': 85.076, 'train_steps_per_second': 5.326, 'total_flos': 213940121334480.0, 'train_loss': 0.38143503889162444, 'epoch': 1.0})

## Avaliando os resultados obtidos

In [ ]:
treinador.evaluate()

{'eval_loss': 0.3713451027870178,
 'eval_f1': 0.8539944903581267,
 'eval_runtime': 4.8402,
 'eval_samples_per_second': 220.24,
 'eval_steps_per_second': 13.842,
 'epoch': 1.0}

## ...mas o que foi treinado afinal de contas?

Vamos exibir a configuração dos parâmetros para entender o que está acontecendo.

In [ ]:
# Exibindo detalhes das camadas
for name, param in modelo.named_parameters():
    print(f"Parameter: {name} ----- {param.requires_grad}")

Parameter: bert.embeddings.word_embeddings.weight ----- True
Parameter: bert.embeddings.position_embeddings.weight ----- True
Parameter: bert.embeddings.token_type_embeddings.weight ----- True
Parameter: bert.embeddings.LayerNorm.weight ----- True
Parameter: bert.embeddings.LayerNorm.bias ----- True
Parameter: bert.encoder.layer.0.attention.self.query.weight ----- True
Parameter: bert.encoder.layer.0.attention.self.query.bias ----- True
Parameter: bert.encoder.layer.0.attention.self.key.weight ----- True
Parameter: bert.encoder.layer.0.attention.self.key.bias ----- True
Parameter: bert.encoder.layer.0.attention.self.value.weight ----- True
Parameter: bert.encoder.layer.0.attention.self.value.bias ----- True
Parameter: bert.encoder.layer.0.attention.output.dense.weight ----- True
Parameter: bert.encoder.layer.0.attention.output.dense.bias ----- True
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.weight ----- True
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.bia

# ⛄ 🥶 🧊❄ Congelamento de Camadas 🧊🧊🧊 ☃ ❄ 🥶

Uma tática muito comum para fine tuning é o congelamento de camadas, ou seja, bloquear a atualização dos pesos no treinamento da rede neural.

O 🐍PyTorch🔥 permite configurar explicitamente quais parâmetros podem ser atualizados. O atributo `requires_grad` é um boolean que controla se o parâmetro requer cálculo de gradiente. Ou seja, quando `requires_grad == False` o sistema de autograd do PyTorch não rastreia as operações que o envolvem, não calcula nem armazena seu gradiente durante o backpropagation, e, consequentemente, o otimizador não ajusta seu valor, mantendo-o constante ao longo de todo o processo de treinamento.

---

Vamos carregar novamente o modelo, mas agora vamos ajustar essa flag apenas nos parâmetros da camada de classificação.



In [ ]:
# Load Model and Tokenizer
modelo_bert_congelado = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

for name, param in modelo_bert_congelado.named_parameters():
    # Ajusta a camada "classifier"
    if name.startswith("classifier"):
      param.requires_grad = True
    else:
      param.requires_grad = False
    print(f"Parameter: {name} ----- {param.requires_grad}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parameter: bert.embeddings.word_embeddings.weight ----- False
Parameter: bert.embeddings.position_embeddings.weight ----- False
Parameter: bert.embeddings.token_type_embeddings.weight ----- False
Parameter: bert.embeddings.LayerNorm.weight ----- False
Parameter: bert.embeddings.LayerNorm.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.query.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.query.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.key.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.key.bias ----- False
Parameter: bert.encoder.layer.0.attention.self.value.weight ----- False
Parameter: bert.encoder.layer.0.attention.self.value.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.weight ----- False
Parameter: bert.encoder.layer.0.attention.output.dense.bias ----- False
Parameter: bert.encoder.layer.0.attention.output.LayerNorm.weight ----- False
Parameter: bert.encoder.layer.0.attention.output

## Treinando o modelo carregado novamente

Agora será executado o processo de treinamento e vejamos a diferença.

In [ ]:
from transformers import TrainingArguments, Trainer

treinador = Trainer(
   model=modelo_bert_congelado,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

treinador.train()

Step,Training Loss
500,0.695700


TrainOutput(global_step=534, training_loss=0.6950046667891941, metrics={'train_runtime': 30.0322, 'train_samples_per_second': 284.029, 'train_steps_per_second': 17.781, 'total_flos': 213940121334480.0, 'train_loss': 0.6950046667891941, 'epoch': 1.0})

## Nova Avaliação do treinamento 🏋

In [ ]:
treinador.evaluate()

{'eval_loss': 0.6907071471214294,
 'eval_f1': 0.6395262768319763,
 'eval_runtime': 4.2383,
 'eval_samples_per_second': 251.517,
 'eval_steps_per_second': 15.808,
 'epoch': 1.0}


---

# Tarefas do Exercício
## 1. Agora responda com suas palavras o que aconteceu em ambos os casos durante o treinamento, evidenciando se você percebeu alguma diferença durante o passo de treinamento. Discuta brevemente os resultados obtidos.

In [ ]:
"""
No primeiro caso, o BERT foi totalmente ajustado (fine-tuning completo), permitindo que tanto o encoder quanto a camada de classificação fossem atualizados.
Já no segundo caso, apenas a camada final de classificação foi treinada, enquanto o encoder permaneceu congelado.
Como consequência, o desempenho foi inferior, pois o modelo não conseguiu adaptar suas representações internas aos dados da tarefa.
"""

' Responda aqui '

## 2. Conforme instruções anteriores, carregue mais uma vez o modelo BERT, mas agora mantenha apenas a partir da camada encoder 10 (`bert.encoder.layer.10`) do modelo BERT como treinável (`require_grad == True`). Ou seja, congele (`requires_grad == False`) até a camada encoder 9 (`bert.encoder.layer.9`).

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_CHECKPOINT = "bert-base-uncased"

model_partial = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

for _, p in model_partial.named_parameters():
    p.requires_grad = False

for n, p in model_partial.named_parameters():
    if n.startswith("classifier"):
        p.requires_grad = True

for layer_id in [10, 11]:
    prefix = f"bert.encoder.layer.{layer_id}."
    for n, p in model_partial.named_parameters():
        if n.startswith(prefix):
            p.requires_grad = True

trainable = [n for n, p in model_partial.named_parameters() if p.requires_grad]

print("Trainable parameters:")

for n in trainable[:20]:
    print(" ", n)

print(f"Total trainable layers: {len(trainable)}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable parameters:
  bert.encoder.layer.10.attention.self.query.weight
  bert.encoder.layer.10.attention.self.query.bias
  bert.encoder.layer.10.attention.self.key.weight
  bert.encoder.layer.10.attention.self.key.bias
  bert.encoder.layer.10.attention.self.value.weight
  bert.encoder.layer.10.attention.self.value.bias
  bert.encoder.layer.10.attention.output.dense.weight
  bert.encoder.layer.10.attention.output.dense.bias
  bert.encoder.layer.10.attention.output.LayerNorm.weight
  bert.encoder.layer.10.attention.output.LayerNorm.bias
  bert.encoder.layer.10.intermediate.dense.weight
  bert.encoder.layer.10.intermediate.dense.bias
  bert.encoder.layer.10.output.dense.weight
  bert.encoder.layer.10.output.dense.bias
  bert.encoder.layer.10.output.LayerNorm.weight
  bert.encoder.layer.10.output.LayerNorm.bias
  bert.encoder.layer.11.attention.self.query.weight
  bert.encoder.layer.11.attention.self.query.bias
  bert.encoder.layer.11.attention.self.key.weight
  bert.encoder.layer.11.at

## 3. Execute o treinamento com os mesmos parâmetros utilizados anteriormente e execute o método `evaluate()` para calcular as métricas e discuta os resultados obtidos.
Caso julgar necessário, execute outros treinamentos ajustando quantidades distintas de parâmetros treináveis para tirar conclusões adicionais.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    "model_partial",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model_partial,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
metrics


Step,Training Loss
500,0.463900


{'eval_loss': 0.40773797035217285,
 'eval_f1': 0.8222222222222222,
 'eval_runtime': 3.7788,
 'eval_samples_per_second': 282.098,
 'eval_steps_per_second': 17.73,
 'epoch': 1.0}

## 4. (BÔNUS) Pesquise e escolha outro dataset para fazer fine tuning do modelo BERT.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
DATASET = ("tweet_eval", "sentiment")
CHECKPOINT = "bert-base-uncased"
MAX_LEN = 128
EPOCHS = 1
BS = 16
LR = 2e-5

In [ ]:
def load_ds(spec):
    if isinstance(spec, tuple):
        return load_dataset(spec[0], spec[1])

    return load_dataset(spec)

raw = load_ds(DATASET)
tok = AutoTokenizer.from_pretrained(CHECKPOINT)

def tok_fn(b):
    return tok(b["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
ds = raw.map(tok_fn, batched=True)
ds = ds.rename_column("label", "labels")
ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

num_labels = ds["train"].features["labels"].num_classes
model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=num_labels)

args = TrainingArguments(
    output_dir="bert_bonus",
    learning_rate=LR,
    per_device_train_batch_size=BS,
    per_device_eval_batch_size=BS,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    tokenizer=tok,
    data_collator=data_collator,      # reuse from notebook
    compute_metrics=compute_metrics,  # reuse from notebook
)

trainer.train()
trainer.evaluate()

Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss


KeyboardInterrupt: 